# Training Data Setup

This notebook prepares training, validation, and test datasets for ML models:
1. Load input (temperature) and target (CRE) data
2. Split by time periods (train/val/test)
3. Split by ensemble members
4. Create hemisphere-based splits for regional analysis
5. Save processed splits for model training

In [ ]:
import sys
sys.path.append('..')

from utils import (
    load_dataset,
    select_variable,
    align_datasets,
    subset_time,
    subset_latitude,
    create_time_period_masks,
    create_hemisphere_masks,
    save_dataset
)
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
# Load input (surface temperature) and target (CRE) datasets
x_data = load_dataset("../../data/CanESM5_1850-2100_tas.nc")
y_data_rsut = load_dataset("../../data/CanESM5_1850-2100_rsutcre.nc")
y_data_rlut = load_dataset("../../data/CanESM5_1850-2100_rlutcre.nc")

print("Input dataset (SST):")
print(x_data)
print("\nTarget dataset (Shortwave CRE):")
print(y_data_rsut)
print("\nTarget dataset (Longwave CRE):")
print(y_data_rlut)

In [ ]:
# Select variables
x_var = select_variable(x_data, ["tas"])
y_var_rsut = select_variable(y_data_rsut, ["cre", "rsutcre"])
y_var_rlut = select_variable(y_data_rlut, ["cre", "rlutcre"])

x_da = x_data[x_var]
y_da_rsut = y_data_rsut[y_var_rsut]
y_da_rlut = y_data_rlut[y_var_rlut]

print(f"\nSelected variables:")
print(f"  Input: {x_var}")
print(f"  Target (SW): {y_var_rsut}")
print(f"  Target (LW): {y_var_rlut}")

In [ ]:
# Align datasets to common coordinates
x_da, y_da_rsut, y_da_rlut = align_datasets(x_da, y_da_rsut, y_da_rlut, join="inner")

print("Aligned datasets:")
print(f"  x: {x_da.dims}")
print(f"  y_rsut: {y_da_rsut.dims}")
print(f"  y_rlut: {y_da_rlut.dims}")

## 2. Time-Based Splitting

Split data into:
- **Train**: 1850-2014 (historical period)
- **Validation**: 1850-2014 (same period, different members)
- **Test**: 2015-2100 (future projections)

In [ ]:
# Verify member dimension exists
if "member" not in x_da.dims:
    raise ValueError("Expected 'member' dimension in input data")

n_members = x_da.sizes["member"]
print(f"Total ensemble members: {n_members}")

In [ ]:
# Create time masks
years = x_da["time"].dt.year
train_mask = (years >= 1850) & (years <= 2014)
val_mask = (years >= 1850) & (years <= 2014)  # Same period, different members
test_mask = years >= 2015

print(f"Time periods:")
print(f"  Train/Val: 1850-2014 ({train_mask.sum().values} timesteps)")
print(f"  Test: 2015+ ({test_mask.sum().values} timesteps)")

In [ ]:
# Random member split (reproducible)
rng = np.random.default_rng(42)
member_idx = rng.permutation(n_members)

train_idx = member_idx[:17]  # 68% for training
val_idx = member_idx[17:21]  # 16% for validation
test_idx = member_idx[21:]   # 16% for testing

print(f"\nMember splits:")
print(f"  Train: {len(train_idx)} members - {train_idx}")
print(f"  Val: {len(val_idx)} members - {val_idx}")
print(f"  Test: {len(test_idx)} members - {test_idx}")

In [ ]:
# Create train/val/test splits for input (temperature)
x_train_period = subset_time(x_da, "time", train_mask)
x_val_period = subset_time(x_da, "time", val_mask)
x_test_period = subset_time(x_da, "time", test_mask)

X_train = x_train_period.isel(member=train_idx)
X_val = x_val_period.isel(member=val_idx)
X_test = x_test_period.isel(member=test_idx)

print("\nInput (X) splits:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

In [ ]:
# Create train/val/test splits for targets (CRE)
# Shortwave CRE
y_rsut_train_period = subset_time(y_da_rsut, "time", train_mask)
y_rsut_val_period = subset_time(y_da_rsut, "time", val_mask)
y_rsut_test_period = subset_time(y_da_rsut, "time", test_mask)

y_train_rsut = y_rsut_train_period.isel(member=train_idx)
y_val_rsut = y_rsut_val_period.isel(member=val_idx)
y_test_rsut = y_rsut_test_period.isel(member=test_idx)

# Longwave CRE
y_rlut_train_period = subset_time(y_da_rlut, "time", train_mask)
y_rlut_val_period = subset_time(y_da_rlut, "time", val_mask)
y_rlut_test_period = subset_time(y_da_rlut, "time", test_mask)

y_train_rlut = y_rlut_train_period.isel(member=train_idx)
y_val_rlut = y_rlut_val_period.isel(member=val_idx)
y_test_rlut = y_rlut_test_period.isel(member=test_idx)

print("\nTarget (y) splits - Shortwave CRE:")
print(f"  y_train_rsut: {y_train_rsut.shape}")
print(f"  y_val_rsut: {y_val_rsut.shape}")
print(f"  y_test_rsut: {y_test_rsut.shape}")

print("\nTarget (y) splits - Longwave CRE:")
print(f"  y_train_rlut: {y_train_rlut.shape}")
print(f"  y_val_rlut: {y_val_rlut.shape}")
print(f"  y_test_rlut: {y_test_rlut.shape}")

## 3. Period-Based Splits

Create additional splits by time period for temporal generalization analysis:
- **Past**: 1850-1950
- **Present**: 1950-2015
- **Future**: 2015-2100

In [ ]:
# Create period masks
time_masks = create_time_period_masks(
    x_da["time"],
    past_range=(1850, 1950),
    present_range=(1950, 2015),
    future_range=(2015, 2100)
)

# Apply to all datasets
period_splits = {}
for period_name, mask in time_masks.items():
    period_splits[period_name] = {
        'X': subset_time(x_da, 'time', mask),
        'y_rsut': subset_time(y_da_rsut, 'time', mask),
        'y_rlut': subset_time(y_da_rlut, 'time', mask)
    }

print("\nPeriod splits:")
for period_name, data in period_splits.items():
    n = data['X'].sizes['time']
    print(f"  {period_name}: {n} timesteps")

## 4. Hemisphere-Based Splits

Create regional splits for analyzing model performance by latitude:
- **Northern**: lat > 23.5°N
- **Equatorial**: -23.5° ≤ lat ≤ 23.5°
- **Southern**: lat < -23.5°S

In [ ]:
# Create hemisphere masks
lat_coord = x_da['lat']
hemisphere_masks = create_hemisphere_masks(lat_coord)

# Apply to all datasets
hemisphere_splits = {}
for region_name, mask in hemisphere_masks.items():
    hemisphere_splits[region_name] = {
        'X': subset_latitude(x_da, mask),
        'y_rsut': subset_latitude(y_da_rsut, mask),
        'y_rlut': subset_latitude(y_da_rlut, mask)
    }

print("\nHemisphere splits:")
for region_name, data in hemisphere_splits.items():
    n = data['X'].sizes['lat']
    print(f"  {region_name}: {n} latitude points")

## 5. Save Processed Splits

Save all splits for use in model training.

In [ ]:
# Save main train/val/test splits
output_dir = "../../data/splits/"

# Input splits
save_dataset(X_train, f"{output_dir}X_train.nc")
save_dataset(X_val, f"{output_dir}X_val.nc")
save_dataset(X_test, f"{output_dir}X_test.nc")

# Shortwave CRE targets
save_dataset(y_train_rsut, f"{output_dir}y_train_rsut.nc")
save_dataset(y_val_rsut, f"{output_dir}y_val_rsut.nc")
save_dataset(y_test_rsut, f"{output_dir}y_test_rsut.nc")

# Longwave CRE targets
save_dataset(y_train_rlut, f"{output_dir}y_train_rlut.nc")
save_dataset(y_val_rlut, f"{output_dir}y_val_rlut.nc")
save_dataset(y_test_rlut, f"{output_dir}y_test_rlut.nc")

print("\n✓ Saved main train/val/test splits")

In [ ]:
# Save period splits
for period_name, data in period_splits.items():
    save_dataset(data['X'], f"{output_dir}period_{period_name}_X.nc")
    save_dataset(data['y_rsut'], f"{output_dir}period_{period_name}_y_rsut.nc")
    save_dataset(data['y_rlut'], f"{output_dir}period_{period_name}_y_rlut.nc")

print("✓ Saved period splits")

In [ ]:
# Save hemisphere splits
for region_name, data in hemisphere_splits.items():
    save_dataset(data['X'], f"{output_dir}hemisphere_{region_name}_X.nc")
    save_dataset(data['y_rsut'], f"{output_dir}hemisphere_{region_name}_y_rsut.nc")
    save_dataset(data['y_rlut'], f"{output_dir}hemisphere_{region_name}_y_rlut.nc")

print("✓ Saved hemisphere splits")

## Summary

Created and saved the following data splits:

### Main Splits (by member and time)
- Train: 17 members × 1850-2014
- Validation: 4 members × 1850-2014
- Test: 4 members × 2015-2100

### Period Splits
- Past: 1850-1950
- Present: 1950-2015
- Future: 2015-2100

### Hemisphere Splits
- Northern (> 23.5°N)
- Equatorial (-23.5° to 23.5°)
- Southern (< -23.5°S)

All splits include both input (surface temperature) and targets (shortwave & longwave CRE).